# §4 — RESISC45 fine-tuning: linear probe vs LoRA vs full FT

Runs `scripts/finetune_resisc.py` for each of the three methods (linear probe → LoRA r=8, α=16 → full FT) and prints the comparison table. Each method trains for 10 epochs at the shared default LR (`optim.lr` from `configs/lora_resisc.yaml`); to tune a method that lags badly, add `'lr': <value>` to its entry in the `method_specs` list and report the tuning in the writeup.

**Prereq:** `runs/clip_eurosat/best.pt` from §3 must already exist on Drive.

**Before running:**
1. Switch the Colab runtime to a GPU (L4 / A100 recommended).
2. Put the `hw3` repo on Google Drive (or clone it locally inside Colab) and edit `REPO_ROOT` below.

## 1. Install dependencies

In [1]:
%%capture
# Do NOT upgrade torch/torchvision: Colab's preinstalled pair is ABI-matched
# (and CUDA-enabled). Upgrading them produces
# `RuntimeError: operator torchvision::nms does not exist`.
# Do NOT pin numpy<2: Colab's preinstalled torch/torchvision are built against
# numpy 2.x, and downgrading produces
# `ValueError: numpy.dtype size changed, may indicate binary incompatibility`.
!pip -q install -U transformers datasets "sentence-transformers<4.0" accelerate tqdm matplotlib pyyaml einops
# Force a clean Pillow install — Colab's preinstalled Pillow can have stale files
# after upgrade, causing `ImportError: cannot import name '_Ink' from 'PIL._typing'`.
!pip -q install --force-reinstall --no-deps --no-cache-dir pillow==11.3.0

## 2. Mount Drive and set up the repo path

In [6]:
import os, sys
from pathlib import Path

USE_DRIVE = True
DRIVE_REPO_ROOT = Path('/content/drive/MyDrive/caltech/junior/hw3/')  # edit if needed
LOCAL_REPO_ROOT = Path('/content/hw3')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    REPO_ROOT = DRIVE_REPO_ROOT
else:
    REPO_ROOT = LOCAL_REPO_ROOT

assert REPO_ROOT.exists(), f'Repo root does not exist: {REPO_ROOT}'
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

Mounted at /content/drive
cwd = /content/drive/MyDrive/caltech/junior/hw3


In [7]:
import torch, torchvision

print('torch:', torch.__version__, '| torchvision:', torchvision.__version__, '| cuda:', torch.cuda.is_available())

# Will raise if torch <-> torchvision ABI is broken.
torchvision.ops.nms(torch.zeros(0, 4), torch.zeros(0), 0.5)
print('torchvision ops OK')

torch: 2.10.0+cu128 | torchvision: 0.25.0+cu128 | cuda: True
torchvision ops OK


## 3. Configure paths and confirm the §3 checkpoint exists

In [9]:
import yaml
from pathlib import Path

CONFIG_PATH = REPO_ROOT / 'configs' / 'lora_resisc.yaml'
PRETRAINED = REPO_ROOT / 'runs' / 'clip_eurosat' / 'best.pt'

assert CONFIG_PATH.exists(), f'config missing: {CONFIG_PATH}'
assert PRETRAINED.exists(), (
    f'CLIP-pretrained checkpoint missing at {PRETRAINED}. '
    'Run §3 (clip_pretrain.ipynb) first and sync best.pt to Drive.'
)

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print(f'config : {CONFIG_PATH}')
print(f'ckpt   : {PRETRAINED}  ({PRETRAINED.stat().st_size / 1024**2:.1f} MB)')
print(f'shared lr (cfg.optim.lr) = {cfg["optim"]["lr"]}')
print(f'epochs   (cfg.train.num_epochs) = {cfg["train"]["num_epochs"]}')

config : /content/drive/MyDrive/caltech/junior/hw3/configs/lora_resisc.yaml
ckpt   : /content/drive/MyDrive/caltech/junior/hw3/runs/clip_eurosat/best.pt  (41.8 MB)
shared lr (cfg.optim.lr) = 0.0001
epochs   (cfg.train.num_epochs) = 10


## 4. Run all three methods

Routes each method's `metrics.json` to local Colab storage (Drive FUSE drops under high-frequency writes), then single-shot copies the result to `REPO_ROOT/runs/resisc_<method>/`. Per-run wall-clock, peak GPU memory (`torch.cuda.max_memory_allocated`), trainable param count, and final test accuracy are saved into `metrics.json` and printed.

To tune the LR for a method that lags badly, add `'lr': <value>` to its dict below. The chosen LR shows up in the summary table, so the tuning is auditable.

In [12]:
import importlib
import basics.lora, basics.vit, scripts.finetune_resisc

# Order matters: reload dependencies before consumers.
for m in (basics.lora, basics.vit, scripts.finetune_resisc):
    importlib.reload(m)

# Re-bind the names the train cell will use.
from scripts.finetune_resisc import train_one, print_table, _default_run_dir

In [13]:
import argparse
import shutil
import time

from scripts.finetune_resisc import train_one, print_table, _default_run_dir

method_specs = [
    # {'method': 'linear_probe'},
    {'method': 'lora', 'rank': 8, 'alpha': 16.0},
    {'method': 'full_ft'},
    # Example of per-method LR tuning (mention it in the writeup if used):
    # {'method': 'full_ft', 'lr': 5.0e-5},
]

LOCAL_RUNS = Path('/content/runs')
LOCAL_RUNS.mkdir(parents=True, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

all_metrics = {}
for spec in method_specs:
    method = spec['method']
    rank = spec.get('rank', 8)
    alpha = spec.get('alpha', 16.0)
    lr_override = spec.get('lr', None)

    local_dir = _default_run_dir(LOCAL_RUNS, method, rank)
    local_dir.mkdir(parents=True, exist_ok=True)

    args = argparse.Namespace(
        config=CONFIG_PATH,
        method=method,
        rank=rank,
        alpha=alpha,
        lr=lr_override,
        pretrained=PRETRAINED,
        output_dir=local_dir,
        device=device,
        summarize=False,
        runs_root=LOCAL_RUNS,
    )

    print(f'\n========== {method} (rank={rank}, lr_override={lr_override}) ==========')
    t0 = time.time()
    metrics = train_one(args, cfg)
    print(f'[{method}] wall (incl. eval) = {time.time() - t0:.1f}s')
    all_metrics[method] = metrics

    # Sync metrics.json back to Drive (single-shot write).
    drive_dir = REPO_ROOT / 'runs' / local_dir.name
    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local_dir / 'metrics.json', drive_dir / 'metrics.json')
    print(f'synced metrics.json -> {drive_dir}')

print('\n========== summary ==========')
print_table(all_metrics)


========== lora (rank=8, lr_override=None) ==========


[lora] epoch 1/10:   0%|          | 0/147 [00:00<?, ?it/s]

[lora] epoch 1/10  train_loss=3.6616  test_acc=0.1738


[lora] epoch 2/10:   0%|          | 0/147 [00:00<?, ?it/s]

[lora] epoch 2/10  train_loss=2.9458  test_acc=0.2876


...

[full_ft] epoch 9/10:   0%|          | 0/147 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7911206e4680><function _MultiProcessingDataLoaderIter.__del__ at 0x7911206e4680>Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7911206e4680>

<function _MultiProcessingDataLoaderIter.__del__ at 0x7911206e4680>
Traceback (most recent call last):
Traceback (most recent call last):

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
            self._shutdown_workers()    self._shutdown_workers()self._shutdown_workers()

[full_ft] epoch 9/10  train_loss=0.7180  test_acc=0.6600


[full_ft] epoch 10/10:   0%|          | 0/147 [00:00<?, ?it/s]

[full_ft] epoch 10/10  train_loss=0.6910  test_acc=0.6603
saved /content/runs/resisc_full_ft/metrics.json
[full_ft] wall (incl. eval) = 229.9s
synced metrics.json -> /content/drive/MyDrive/caltech/junior/hw3/runs/resisc_full_ft

========== summary ==========

        method |        lr |   test_acc |      trainable |  peak_mem_MB |     time_s
------------------------------------------------------------------------------------
          lora |     1e-04 |     0.4198 |        275,373 |       1108.5 |      161.1
       full_ft |     1e-04 |     0.6603 |     10,755,117 |       1629.9 |      206.2



## 5. Re-print the comparison table via the `--summarize` action

Reads `metrics.json` from each method's run directory and prints the same table. Useful after a kernel restart, or to regenerate the deliverable without retraining.

In [14]:
import json
from scripts.finetune_resisc import print_table, _default_run_dir

RANK = 8  # only affects the lora run directory name
saved_metrics = {}
for method in ('linear_probe', 'lora', 'full_ft'):
    p = _default_run_dir(REPO_ROOT / 'runs', method, RANK) / 'metrics.json'
    if p.exists():
        saved_metrics[method] = json.loads(p.read_text())
    else:
        print(f'[warn] missing {p}')
print_table(saved_metrics)


        method |        lr |   test_acc |      trainable |  peak_mem_MB |     time_s
------------------------------------------------------------------------------------
  linear_probe |     1e-04 |     0.2681 |         17,325 |        205.1 |      112.4
          lora |     1e-04 |     0.4198 |        275,373 |       1108.5 |      161.1
       full_ft |     1e-04 |     0.6603 |     10,755,117 |       1629.9 |      206.2



## 6. Writeup notes (4–5 sentences)

Fill in after the table renders. Things to cover:
- **Trainable parameters**: linear probe (~17k) vs LoRA r=8 (~275k = ~258k adapter + head) vs full FT (~11M). Cite the actual numbers from the table.
- **Peak GPU memory**: tracks trainable params, *not* total params — frozen layers don't store grads or optimizer state. Expect linear probe ≈ LoRA ≪ full FT.
- **Wall-clock**: linear probe and LoRA still run the full backbone forward/backward (gradients have to flow back to the head/adapters), so time savings vs. full FT come mainly from a smaller optimizer step and reduced grad allocation — typically modest (~1.1–1.3×), not dramatic.
- **Test accuracy**: full FT usually wins by a few points on RESISC45's 45 fine-grained classes; LoRA typically closes most of that gap; the linear probe lags more, because CLIP-on-EuroSAT features don't fully cover RESISC's category granularity.
- **Trade-off framing**: LoRA is the Pareto choice — ~40× fewer trainable params than full FT, ~linear-probe memory, accuracy close to full FT.

If you used a per-method LR override, note it here: e.g., "full FT diverged at the shared 1e-4; retrained at 5e-5 — that's the row in the table."